In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
@torch.no_grad()
def build_candidate_vocab(token_ids_BN: torch.Tensor, vocab_size: int, cand_size: int = 4096):
    """
    token_ids_BN: (B,N) long
    returns:
      cand_vocab: (Vcand,) long  (必ずバッチ内正例を含む)
    """
    device = token_ids_BN.device
    pos = torch.unique(token_ids_BN)

    # pos が cand_size を超えるケースは稀だが、起きたら pos のみ（= cand_size以上）で処理
    # その場合は Vcand=pos.numel() になり、候補語彙が大きくなる（正確性優先）
    if pos.numel() >= cand_size:
        return pos

    need = cand_size - pos.numel()
    neg = torch.randint(0, vocab_size, (need,), device=device, dtype=torch.long)
    cand = torch.unique(torch.cat([pos, neg], dim=0))
    # もし unique で減って cand_size を下回ったら追加（簡易ループ）
    while cand.numel() < cand_size:
        extra = torch.randint(0, vocab_size, (cand_size - cand.numel(),), device=device, dtype=torch.long)
        cand = torch.unique(torch.cat([cand, extra], dim=0))

    # 上回ったらランダムに削る（ただし pos は必ず残す）
    if cand.numel() > cand_size:
        # pos を必ず含むように：cand から pos を引いた残りをシャッフルして詰める
        mask_pos = torch.isin(cand, pos)
        keep_pos = cand[mask_pos]
        rest = cand[~mask_pos]
        take = cand_size - keep_pos.numel()
        rest = rest[torch.randperm(rest.numel(), device=device)[:take]]
        cand = torch.cat([keep_pos, rest], dim=0)

    return cand

def slot_logits_chunk_cand(self, slots, pos_s, n0, n1, cand_vocab):
    """
    cand_vocab: (Vcand,) long
    returns slot_logits: (B,K,M,Vcand)
    """
    slot_pos = slots[:, :, None, :] + pos_s[:, None, n0:n1, :]  # (B,K,M,S)
    h = self.dec(slot_pos)                                      # (B,K,M,E)

    # weight tying: logits = h @ Emb[cand]^T
    W = self.emb.weight[cand_vocab]                             # (Vcand,E)
    logits = torch.einsum("bkme,ve->bkmv", h, W)                # (B,K,M,Vcand)
    return logits

# ---- Slot Attention (前と同じ) ----
class SlotAttention(nn.Module):
    def __init__(self, num_slots: int, in_dim: int, slot_dim: int,
                 iters: int = 3, eps: float = 1e-8, hidden_dim: int = 128):
        super().__init__()
        self.num_slots = num_slots
        self.iters = iters
        self.eps = eps
        self.scale = slot_dim ** -0.5

        self.norm_inputs = nn.LayerNorm(in_dim)
        self.norm_slots  = nn.LayerNorm(slot_dim)
        self.norm_mlp    = nn.LayerNorm(slot_dim)

        self.slots_mu = nn.Parameter(torch.zeros(1, 1, slot_dim))
        self.slots_sigma = nn.Parameter(torch.ones(1, 1, slot_dim))

        self.to_q = nn.Linear(slot_dim, slot_dim, bias=False)
        self.to_k = nn.Linear(in_dim,   slot_dim, bias=False)
        self.to_v = nn.Linear(in_dim,   slot_dim, bias=False)

        self.gru = nn.GRUCell(slot_dim, slot_dim)
        self.mlp = nn.Sequential(
            nn.Linear(slot_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, slot_dim),
        )

    def forward(self, inputs):
        B, N, D = inputs.shape
        inputs = self.norm_inputs(inputs)

        mu = self.slots_mu.expand(B, self.num_slots, -1)
        sigma = self.slots_sigma.expand(B, self.num_slots, -1)
        slots = mu + sigma * torch.randn_like(mu)

        k = self.to_k(inputs)
        v = self.to_v(inputs)

        for _ in range(self.iters):
            slots_prev = slots
            q = self.to_q(self.norm_slots(slots))

            attn_logits = torch.einsum("bkd,bnd->bkn", q, k) * self.scale
            attn = F.softmax(attn_logits, dim=1)  # over slots
            attn = attn + self.eps
            attn = attn / attn.sum(dim=-1, keepdim=True)

            updates = torch.einsum("bkn,bnd->bkd", attn, v)
            slots = self.gru(
                updates.reshape(B*self.num_slots, -1),
                slots_prev.reshape(B*self.num_slots, -1),
            ).reshape(B, self.num_slots, -1)
            slots = slots + self.mlp(self.norm_mlp(slots))

        return slots, attn  # attn: (B,K,N)

# ---- pos emb ----
def build_2d_sincos_pos_embed(h, w, dim, device=None):
    assert dim % 4 == 0
    y = torch.arange(h, device=device).float()
    x = torch.arange(w, device=device).float()
    yy, xx = torch.meshgrid(y, x, indexing="ij")

    omega = torch.arange(dim // 4, device=device).float()
    omega = 1.0 / (10000 ** (omega / (dim // 4)))

    out_y = yy[..., None] * omega[None, None, :]
    out_x = xx[..., None] * omega[None, None, :]
    pos = torch.cat([torch.sin(out_y), torch.cos(out_y),
                     torch.sin(out_x), torch.cos(out_x)], dim=-1)
    return pos  # (H,W,dim)

class TokenSlotCE(nn.Module):
    """
    Token IDs (B,3,32,32) -> SlotAttention -> per-slot logits -> masked mixture -> CE
    """
    def __init__(
        self,
        vocab_size: int,
        emb_dim: int = 256,
        num_slots: int = 8,
        slot_dim: int = 256,
        iters: int = 3,
        T: int = 3, H: int = 32, W: int = 32,
        dec_hidden: int = 512,
    ):
        super().__init__()
        self.vocab_size = int(vocab_size)
        self.emb_dim = int(emb_dim)
        self.num_slots = int(num_slots)
        self.slot_dim = int(slot_dim)
        self.T, self.H, self.W = int(T), int(H), int(W)
        self.N = self.T * self.H * self.W

        self.emb = nn.Embedding(self.vocab_size, self.emb_dim)
        self.time_emb = nn.Embedding(self.T, self.emb_dim)
        self.register_buffer("pos2d", build_2d_sincos_pos_embed(self.H, self.W, self.emb_dim), persistent=False)

        self.pre = nn.Sequential(
            nn.LayerNorm(self.emb_dim),
            nn.Linear(self.emb_dim, self.emb_dim),
            nn.ReLU(inplace=True),
            nn.Linear(self.emb_dim, self.emb_dim),
        )

        self.slot_attn = SlotAttention(num_slots=self.num_slots, in_dim=self.emb_dim, slot_dim=self.slot_dim, iters=iters)

        # decoder: slot + pos -> hidden -> logits (vocab)
        self.dec_pos = nn.Linear(self.emb_dim, self.slot_dim, bias=False)
        self.dec = nn.Sequential(
            nn.LayerNorm(self.slot_dim),
            nn.Linear(self.slot_dim, dec_hidden),
            nn.ReLU(inplace=True),
            nn.Linear(dec_hidden, self.emb_dim),
        )
        # 語彙射影（重いので weight tying も可）
        self.to_vocab = nn.Linear(self.emb_dim, self.vocab_size, bias=False)

    def encode(self, token_frames: torch.Tensor):
        """
        token_frames: (B,T,H,W) int64
        returns:
          slots: (B,K,S)
          attn : (B,K,N)
          pos_s: (B,N,S)  # decoder用
        """
        B, T, H, W = token_frames.shape
        assert (T, H, W) == (self.T, self.H, self.W)

        x = self.emb(token_frames)  # (B,T,H,W,E)
        x = x + self.pos2d[None, None, :, :, :]
        t_ids = torch.arange(T, device=token_frames.device)
        x = x + self.time_emb(t_ids)[None, :, None, None, :]

        x = self.pre(x).reshape(B, self.N, -1)  # (B,N,E)
        slots, attn = self.slot_attn(x)         # (B,K,S), (B,K,N)

        # decoder pos in slot space
        pos = self.pos2d[None, :, :, :].expand(B, H, W, -1).reshape(B, H*W, -1)     # (B,1024,E)
        pos = pos[:, None, :, :].expand(B, T, H*W, -1).reshape(B, self.N, -1)       # (B,N,E)
        pos_s = self.dec_pos(pos)  # (B,N,S)
        return slots, attn, pos_s

    def slot_logits_chunk(self, slots: torch.Tensor, pos_s: torch.Tensor, n0: int, n1: int):
        """
        slots: (B,K,S)
        pos_s: (B,N,S)
        returns slot_logits: (B,K,(n1-n0),V)
        """
        # (B,K,1,S) + (B,1,M,S) -> (B,K,M,S)
        slot_pos = slots[:, :, None, :] + pos_s[:, None, n0:n1, :]
        h = self.dec(slot_pos)               # (B,K,M,E)
        logits = self.to_vocab(h)            # (B,K,M,V)
        return logits

    def forward_loss(self, token_frames: torch.Tensor, chunk_n: int = 256):
        """
        正確な full-softmax CE を chunked に計算
        returns: loss (scalar), attn (B,K,N) [debug用]
        """
        B, T, H, W = token_frames.shape
        slots, attn, pos_s = self.encode(token_frames)  # attn: (B,K,N)

        target = token_frames.reshape(B, self.N)  # (B,N)

        total_loss = 0.0
        total_count = 0

        # log(attn) を足して mixture を logsumexp で作る
        log_attn = (attn + 1e-8).log()  # (B,K,N)

        for n0 in range(0, self.N, chunk_n):
            n1 = min(self.N, n0 + chunk_n)
            M = n1 - n0

            slot_logits = self.slot_logits_chunk(slots, pos_s, n0, n1)  # (B,K,M,V)
            # log p_k(v|n)
            slot_logprob = F.log_softmax(slot_logits, dim=-1)           # (B,K,M,V)

            # mixture: log Σ_k exp(log_attn + logprob)
            mix_logprob = torch.logsumexp(
                log_attn[:, :, n0:n1].unsqueeze(-1) + slot_logprob, dim=1
            )  # (B,M,V)

            # CE: -log p(target)
            tgt = target[:, n0:n1]  # (B,M)
            nll = -mix_logprob.gather(dim=-1, index=tgt.unsqueeze(-1)).squeeze(-1)  # (B,M)

            total_loss = total_loss + nll.sum()
            total_count += tgt.numel()

        loss = total_loss / total_count
        return loss, attn

    def forward_loss_candidate_ce(self, token_frames: torch.Tensor, cand_size: int = 4096, chunk_n: int = 256):
        """
        Restricted softmax CE (candidate vocab).
        returns: loss (scalar), attn (B,K,N)
        """
        B, T, H, W = token_frames.shape
        slots, attn, pos_s = self.encode(token_frames)  # attn:(B,K,N)

        target = token_frames.reshape(B, self.N)        # (B,N)

        # ★候補語彙（バッチ内正例は必ず含む）
        cand_vocab = build_candidate_vocab(target, self.vocab_size, cand_size=cand_size)  # (Vcand,)
        Vc = cand_vocab.numel()

        log_attn = (attn + 1e-8).log()  # (B,K,N)

        total_loss = 0.0
        total_count = 0

        # 正例 token_id -> cand index を作る（全N分まとめて）
        # eq: (B,N,Vc) は重いので、ここは “ソート+search” にするのが理想だが、
        # Vcが4k程度なら B*N=~3k*B でも許容になりがち。より軽い実装は下の注釈参照。
        eq = (target.unsqueeze(-1) == cand_vocab.view(1, 1, -1))  # (B,N,Vc)
        if not bool(eq.any(dim=-1).all().item()):
            raise RuntimeError("Some targets are not included in candidate vocab. Increase cand_size.")
        tgt_idx_all = eq.float().argmax(dim=-1)  # (B,N) cand内index

        for n0 in range(0, self.N, chunk_n):
            n1 = min(self.N, n0 + chunk_n)
            M = n1 - n0

            # (B,K,M,Vc)
            slot_logits = self.slot_logits_chunk_cand(slots, pos_s, n0, n1, cand_vocab)
            slot_logprob = F.log_softmax(slot_logits, dim=-1)  # (B,K,M,Vc)

            mix_logprob = torch.logsumexp(
                log_attn[:, :, n0:n1].unsqueeze(-1) + slot_logprob,
                dim=1
            )  # (B,M,Vc)

            tgt_idx = tgt_idx_all[:, n0:n1]  # (B,M) cand index
            nll = -mix_logprob.gather(-1, tgt_idx.unsqueeze(-1)).squeeze(-1)  # (B,M)

            total_loss = total_loss + nll.sum()
            total_count += tgt_idx.numel()

        loss = total_loss / total_count
        return loss, attn, Vc


In [2]:
import sys, os
sys.path.append(os.path.abspath('../src/'))
from dataset import TrainDataset

ds = TrainDataset(
    root="/root/work/data/raw/train_v2.0",
    output_format="seq2seq",
    cache_path="/root/work/data/outputs/valid_starts_stride3_clipclean.npy",
)

In [ ]:
from torch.utils.data import DataLoader
from tqdm import tqdm

def get_past_tensor(batch, device):
    x = batch["past_frames"]
    if isinstance(x, torch.Tensor):
        return x.long().to(device)
    if isinstance(x, np.ndarray):
        return torch.from_numpy(x).long().to(device)
    if isinstance(x, list):
        if isinstance(x[0], torch.Tensor):
            return torch.stack([t.long() for t in x], dim=0).to(device)
        return torch.from_numpy(np.stack(x, axis=0)).long().to(device)
    raise TypeError(type(x))

device = "cuda"

loader = DataLoader(ds, batch_size=2, shuffle=True, num_workers=2, pin_memory=True)

# vocab_size は最大token id+1 に合わせるのが理想
vocab_size = 65536

model = TokenSlotCE(
    vocab_size=vocab_size,
    emb_dim=128,
    num_slots=6,
    slot_dim=128,
    iters=3,
    T=3, H=32, W=32,
).to(device)

opt = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)
scaler = torch.cuda.amp.GradScaler()

model.train()
for epoch in range(1):
    pbar = tqdm(loader, desc=f"epoch {epoch}")
    for batch in pbar:
        tok = get_past_tensor(batch, device)  # (B,3,32,32)

        opt.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(dtype=torch.bfloat16):
            loss, attn = model.forward_loss(tok, chunk_n=128)  # chunk_nはVRAMに応じて 128/256/512

        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()

        pbar.set_postfix(loss=float(loss.detach().cpu()))


/tmp/ipykernel_40690/739109158.py:37: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
epoch 0:   0%|          | 0/309816 [00:00<?, ?it/s]/tmp/ipykernel_40690/739109158.py:46: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.bfloat16):
epoch 0:   0%|          | 35/309816 [11:50<2476:56:51, 28.78s/it, loss=17.2]

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# ---------- utils ----------
@torch.no_grad()
def build_candidate_vocab(token_ids_BN: torch.Tensor, vocab_size: int, cand_size: int = 4096):
    device = token_ids_BN.device
    pos = torch.unique(token_ids_BN)

    if pos.numel() >= cand_size:
        return pos

    need = cand_size - pos.numel()
    neg = torch.randint(0, vocab_size, (need,), device=device, dtype=torch.long)
    cand = torch.unique(torch.cat([pos, neg], dim=0))

    while cand.numel() < cand_size:
        extra = torch.randint(0, vocab_size, (cand_size - cand.numel(),), device=device, dtype=torch.long)
        cand = torch.unique(torch.cat([cand, extra], dim=0))

    if cand.numel() > cand_size:
        mask_pos = torch.isin(cand, pos)
        keep_pos = cand[mask_pos]
        rest = cand[~mask_pos]
        take = cand_size - keep_pos.numel()
        rest = rest[torch.randperm(rest.numel(), device=device)[:take]]
        cand = torch.cat([keep_pos, rest], dim=0)

    return cand

def build_2d_sincos_pos_embed(h, w, dim, device=None):
    assert dim % 4 == 0
    y = torch.arange(h, device=device).float()
    x = torch.arange(w, device=device).float()
    yy, xx = torch.meshgrid(y, x, indexing="ij")

    omega = torch.arange(dim // 4, device=device).float()
    omega = 1.0 / (10000 ** (omega / (dim // 4)))

    out_y = yy[..., None] * omega[None, None, :]
    out_x = xx[..., None] * omega[None, None, :]
    pos = torch.cat([torch.sin(out_y), torch.cos(out_y),
                     torch.sin(out_x), torch.cos(out_x)], dim=-1)
    return pos  # (H,W,dim)

# ---------- Slot Attention ----------
class SlotAttention(nn.Module):
    def __init__(self, num_slots: int, in_dim: int, slot_dim: int,
                 iters: int = 3, eps: float = 1e-8, hidden_dim: int = 128):
        super().__init__()
        self.num_slots = num_slots
        self.iters = iters
        self.eps = eps
        self.scale = slot_dim ** -0.5

        self.norm_inputs = nn.LayerNorm(in_dim)
        self.norm_slots  = nn.LayerNorm(slot_dim)
        self.norm_mlp    = nn.LayerNorm(slot_dim)

        self.slots_mu = nn.Parameter(torch.zeros(1, 1, slot_dim))
        self.slots_sigma = nn.Parameter(torch.ones(1, 1, slot_dim))

        self.to_q = nn.Linear(slot_dim, slot_dim, bias=False)
        self.to_k = nn.Linear(in_dim,   slot_dim, bias=False)
        self.to_v = nn.Linear(in_dim,   slot_dim, bias=False)

        self.gru = nn.GRUCell(slot_dim, slot_dim)
        self.mlp = nn.Sequential(
            nn.Linear(slot_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, slot_dim),
        )

    def forward(self, inputs):
        B, N, _ = inputs.shape
        inputs = self.norm_inputs(inputs)

        mu = self.slots_mu.expand(B, self.num_slots, -1)
        sigma = self.slots_sigma.expand(B, self.num_slots, -1)
        slots = mu + sigma * torch.randn_like(mu)

        k = self.to_k(inputs)
        v = self.to_v(inputs)

        for _ in range(self.iters):
            slots_prev = slots
            q = self.to_q(self.norm_slots(slots))

            attn_logits = torch.einsum("bkd,bnd->bkn", q, k) * self.scale
            attn = F.softmax(attn_logits, dim=1)  # over slots
            attn = attn + self.eps
            attn = attn / attn.sum(dim=-1, keepdim=True)

            updates = torch.einsum("bkn,bnd->bkd", attn, v)
            slots = self.gru(
                updates.reshape(B*self.num_slots, -1),
                slots_prev.reshape(B*self.num_slots, -1),
            ).reshape(B, self.num_slots, -1)
            slots = slots + self.mlp(self.norm_mlp(slots))

        return slots, attn  # (B,K,S), (B,K,N)

# ---------- Candidate-CE Token Slot model ----------
class TokenSlotCE(nn.Module):
    """
    token ids (B,3,32,32) -> slot attention -> per-slot logits over candidate vocab -> mixture -> CE
    """
    def __init__(
        self,
        vocab_size: int,
        emb_dim: int = 256,
        num_slots: int = 8,
        slot_dim: int = 256,
        iters: int = 3,
        T: int = 3, H: int = 32, W: int = 32,
        dec_hidden: int = 512,
    ):
        super().__init__()
        self.vocab_size = int(vocab_size)
        self.emb_dim = int(emb_dim)
        self.num_slots = int(num_slots)
        self.slot_dim = int(slot_dim)
        self.T, self.H, self.W = int(T), int(H), int(W)
        self.N = self.T * self.H * self.W

        self.emb = nn.Embedding(self.vocab_size, self.emb_dim)
        self.time_emb = nn.Embedding(self.T, self.emb_dim)
        self.register_buffer("pos2d", build_2d_sincos_pos_embed(self.H, self.W, self.emb_dim), persistent=False)

        self.pre = nn.Sequential(
            nn.LayerNorm(self.emb_dim),
            nn.Linear(self.emb_dim, self.emb_dim),
            nn.ReLU(inplace=True),
            nn.Linear(self.emb_dim, self.emb_dim),
        )

        self.slot_attn = SlotAttention(num_slots=self.num_slots, in_dim=self.emb_dim, slot_dim=self.slot_dim, iters=iters)

        # slot + pos -> hidden -> emb
        self.dec_pos = nn.Linear(self.emb_dim, self.slot_dim, bias=False)
        self.dec = nn.Sequential(
            nn.LayerNorm(self.slot_dim),
            nn.Linear(self.slot_dim, dec_hidden),
            nn.ReLU(inplace=True),
            nn.Linear(dec_hidden, self.emb_dim),
        )

    def encode(self, token_frames: torch.Tensor):
        B, T, H, W = token_frames.shape
        assert (T, H, W) == (self.T, self.H, self.W)

        x = self.emb(token_frames)  # (B,T,H,W,E)
        x = x + self.pos2d[None, None, :, :, :]
        t_ids = torch.arange(T, device=token_frames.device)
        x = x + self.time_emb(t_ids)[None, :, None, None, :]

        x = self.pre(x).reshape(B, self.N, -1)  # (B,N,E)
        slots, attn = self.slot_attn(x)         # (B,K,S), (B,K,N)

        # pos to slot space
        pos = self.pos2d[None, :, :, :].expand(B, H, W, -1).reshape(B, H*W, -1)     # (B,1024,E)
        pos = pos[:, None, :, :].expand(B, T, H*W, -1).reshape(B, self.N, -1)       # (B,N,E)
        pos_s = self.dec_pos(pos)  # (B,N,S)

        return slots, attn, pos_s

    def slot_logits_chunk_cand(self, slots, pos_s, n0, n1, cand_vocab):
        """
        returns: (B,K,M,Vcand)
        """
        slot_pos = slots[:, :, None, :] + pos_s[:, None, n0:n1, :]  # (B,K,M,S)
        h = self.dec(slot_pos)                                      # (B,K,M,E)

        W = self.emb.weight[cand_vocab]                             # (Vcand,E)
        logits = torch.einsum("bkme,ve->bkmv", h, W)                # (B,K,M,Vcand)
        return logits

    def forward_loss_candidate_ce(self, token_frames: torch.Tensor, cand_size: int = 4096, chunk_n: int = 256):
        B, T, H, W = token_frames.shape
        slots, attn, pos_s = self.encode(token_frames)  # attn:(B,K,N)
        target = token_frames.reshape(B, self.N)        # (B,N)

        cand_vocab = build_candidate_vocab(target, self.vocab_size, cand_size=cand_size)  # (Vcand,)
        Vc = int(cand_vocab.numel())

        # target -> candidate index (簡易版)
        eq = (target.unsqueeze(-1) == cand_vocab.view(1, 1, -1))  # (B,N,Vc)
        if not bool(eq.any(dim=-1).all().item()):
            raise RuntimeError("Some targets missing in candidate vocab. Increase cand_size.")
        tgt_idx_all = eq.float().argmax(dim=-1)  # (B,N)

        log_attn = (attn + 1e-8).log()

        total_loss = 0.0
        total_count = 0

        for n0 in range(0, self.N, chunk_n):
            n1 = min(self.N, n0 + chunk_n)

            slot_logits = self.slot_logits_chunk_cand(slots, pos_s, n0, n1, cand_vocab)  # (B,K,M,Vc)
            slot_logprob = F.log_softmax(slot_logits, dim=-1)                            # (B,K,M,Vc)

            mix_logprob = torch.logsumexp(
                log_attn[:, :, n0:n1].unsqueeze(-1) + slot_logprob, dim=1
            )  # (B,M,Vc)

            tgt_idx = tgt_idx_all[:, n0:n1]  # (B,M)
            nll = -mix_logprob.gather(-1, tgt_idx.unsqueeze(-1)).squeeze(-1)  # (B,M)

            total_loss = total_loss + nll.sum()
            total_count += tgt_idx.numel()

        loss = total_loss / total_count
        return loss, attn, Vc


In [8]:
from torch.utils.data import DataLoader
from tqdm import tqdm
import numpy as np
import torch

def get_past_tensor(batch, device):
    x = batch["past_frames"]
    if isinstance(x, torch.Tensor):
        return x.long().to(device)
    if isinstance(x, np.ndarray):
        return torch.from_numpy(x).long().to(device)
    if isinstance(x, list):
        if isinstance(x[0], torch.Tensor):
            return torch.stack([t.long() for t in x], dim=0).to(device)
        return torch.from_numpy(np.stack(x, axis=0)).long().to(device)
    raise TypeError(type(x))

device = "cuda"
loader = DataLoader(ds, batch_size=4, shuffle=True, num_workers=2, pin_memory=True)
vocab_size = 64030
model = TokenSlotCE(
    vocab_size=vocab_size,
    emb_dim=128,
    num_slots=6,
    slot_dim=128,
    iters=3,
    T=3, H=32, W=32,
).to(device)
model.train()
opt = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)
scaler = torch.cuda.amp.GradScaler()

cand_size = 8192     # まずここから（2048/8192でもOK）
chunk_n = 256        # VRAMに合わせて 128/256/512

for epoch in range(5):
    pbar = tqdm(loader, desc=f"epoch {epoch}")
    for batch in pbar:
        tok = get_past_tensor(batch, device)  # (B,3,32,32)

        opt.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(dtype=torch.bfloat16):
            loss, attn, Vc = model.forward_loss_candidate_ce(tok, cand_size=cand_size, chunk_n=chunk_n)

        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()

        pbar.set_postfix(loss=float(loss.detach().cpu()), Vcand=int(Vc))


/tmp/ipykernel_43622/2793595816.py:31: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
epoch 0:   0%|          | 0/154908 [00:00<?, ?it/s]/tmp/ipykernel_43622/2793595816.py:42: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.bfloat16):
epoch 0:   0%|          | 12/154908 [00:28<102:14:26,  2.38s/it, Vcand=8683, loss=15.6]


KeyboardInterrupt: 